# My First Model: A Guided Classification Walkthrough

Welcome! This notebook will walk you through building your first machine learning model for a **binary classification** problem on tabular data.

This notebook is structured as follows. Aside from the specific models picked for demonstration, this process is pretty standard for tabular data problems.

1. **Load & explore** the data
2. **Visualise** key relationships
3. **Build a baseline** (so we know what "good" looks like)
4. **Preprocess** features (encoding, scaling)
5. **Train/test split** (avoiding data leakage!)
6. **Choose a metric** and understand why
7. **Train a simple model** — Logistic Regression
8. **Train a better model** — LightGBM
9. **Tune hyperparameters** for a quick win
10. **Train final model & generate a submission**

The code is mostly written already - the aim here is to **read the comments, understand what's happening, and fill in the gaps** marked with `# TODO`. Once that's done, you can build upon what's here with your own models, feature engineering, etc.

To begin with, we'll be looking at predicting the payback of a loan, here: https://www.kaggle.com/competitions/playground-series-s5e11.

However, this code - with only a couple of minor tweaks - should work on most other tabular playground binary classification competitions.

Let's go!

---
## 0. Setup & Imports

First, we import the libraries we'll need. These are all pre-installed on Kaggle.

If you're running locally and something is missing, just `pip install <package>`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

print('All imports successful.')

---
## 1. Load the Data

Kaggle competitions typically give you three files:
- `train.csv` — labelled data you train your model on
- `test.csv` — unlabelled data you need to make predictions for
- `sample_submission.csv` — shows you the format your submission needs to be in

The path below assumes you're running this as a **Kaggle Notebook** attached to the competition.
If you're running locally, update the path to wherever you've saved the files.

If we weren't using Kaggle, it would be vital to split the data to have a set of "hold-out" test data. That test data is only used to check the performance of the model as it is "unseen". This should be done before any scaling occurs, and is key to prevent "leakage" - where the model is influenced by the data it is tested on.

In [ ]:
# Update this path if running locally, or if a different comp.
# Note that for live comps '/competitions/' isn't always in the middle
DATA_DIR = '/kaggle/input/competitions/playground-series-s5e11/'

train = pd.read_csv(DATA_DIR + 'train.csv')
test = pd.read_csv(DATA_DIR + 'test.csv')
sample_sub = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print(f'Training set:   {train.shape[0]:,} rows, {train.shape[1]} columns')
print(f'Test set:       {test.shape[0]:,} rows, {test.shape[1]} columns')
print(f'Submission:     {sample_sub.shape[0]:,} rows, {sample_sub.shape[1]} columns')

---
## 2. Explore the Data

Before we touch any modelling, we need to **understand what we're working with**.

Key questions at this stage:
- What columns do we have? What types are they?
- Are there missing values?
- How is the target variable distributed? (Is it balanced or imbalanced?)
- What do the features look like at a glance?

In [ ]:
# Let's start with the basics — what does the data look like?
train.head()

In [ ]:
# Data types and non-null counts — are there any missing values?
train.info()

In [ ]:
# Summary statistics for numeric columns
# Look out for: unexpected min/max values, large ranges, anything that looks odd
train.describe()

In [ ]:
# Summary statistics for categorical columns
train.describe(include='object')

### 2.1 Target Variable Distribution

Understanding the **balance** of the target is crucial. If 99% of rows are class 0,
a model that always predicts 0 gets 99% accuracy — but it's useless!

This is why we need to think carefully about which **metric** we use (more on that later).

In [ ]:
# TODO: Replace 'TARGET_COLUMN' with the actual name of the target column.
# Hint: check the output of train.head() above, or look at sample_submission.csv

TARGET = 'TARGET_COLUMN'

print('Target distribution:')
print(train[TARGET].value_counts())
print(f'\nAs percentages:')
print(train[TARGET].value_counts(normalize=True).round(3) * 100)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
train[TARGET].value_counts().plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'])
ax.set_title('Target Variable Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 3. Understand and Visualise Key Relationships

Visualisation helps us build **intuition** about which features might be useful predictors. However, the data here may not tell the whole story. For any project, a **literature review** is vital to understanding our problem.

**Spend 5-10 mins searching loan payback**;
- How do loan providers usually calculate whether someone is likely to pay back?
- Are our features here similar to those used?
- Are any of our features potentially related to or calculated from each other? Might this be an issue?

You should make sure to understand what each of the features are before modelling. Among many other things (like ability to talk to client SMEs in their language!), this helps us understand what a reasonable baseline model might be, or potentially indicate unexpected biases or ethical questions in our data. There may even be some academic papers where someone has attempted the exact same problem, which can indicate what performance metrics we may expect. 

**TODO**: *Add your findings to this markdown cell*

Now that's complete, we'll make some visualisations of our data. We'll look at:
- Distributions of numeric features, split by the target
- Correlations between numeric features
- Counts of categorical features by target class

In [ ]:
# Identify numeric and categorical columns (excluding id and target)
exclude_cols = ['id', TARGET]
numeric_cols = [c for c in train.select_dtypes(include=np.number).columns if c not in exclude_cols]
categorical_cols = [c for c in train.select_dtypes(include='object').columns if c not in exclude_cols]

print(f'Numeric features ({len(numeric_cols)}):     {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

In [ ]:
# Histograms of numeric features, coloured by target class
# This helps us see if the distributions differ between classes — a good sign for predictive power

n = len(numeric_cols)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label in sorted(train[TARGET].unique()):
        axes[i].hist(
            train.loc[train[TARGET] == label, col].dropna(),
            bins=40, alpha=0.5, label=f'{TARGET}={label}', density=True
        )
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Target Class', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — are any features strongly correlated with each other (or the target)?
# High correlation between features can sometimes cause issues; correlation with the target is useful.

fig, ax = plt.subplots(figsize=(10, 8))
corr = train[numeric_cols + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features: how does the target rate vary across categories?

if categorical_cols:
    n = len(numeric_cols)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    if n == 1:
        axes = [axes]
        
    axes = axes.flatten()

    for i, col in enumerate(categorical_cols):
        rates = train.groupby(col)[TARGET].mean().sort_values()
        rates.plot(kind='barh', ax=axes[i], color='#4C72B0')
        axes[i].set_title(f'Mean {TARGET} by {col}')
        axes[i].set_xlabel(f'Mean {TARGET}')

    plt.tight_layout()
    plt.show()
else:
    print('No categorical columns found.')

---
## 4. Baseline Model

Before building anything fancy, we need a **baseline** — a simple benchmark to beat.

A naive baseline would be to predict the same probability for every row (e.g. the overall
target mean). That gives us a ROC-AUC of ~0.5 — no better than random.

We can do better with a simple **rules-based model** that uses just a single feature.

Look at the correlation heatmap above. Which numeric feature has the **strongest correlation**
(positive or negative) with the target? That feature alone should give us some predictive
power, even without any machine learning.

In [ ]:
# TODO: Based on the correlation heatmap above, identify the numeric feature with the
# strongest absolute correlation to the target. Replace 'BEST_FEATURE' below.

BASELINE_FEATURE = 'BEST_FEATURE'

# Let's check: what is the correlation between this feature and the target?
corr_with_target = train[BASELINE_FEATURE].corr(train[TARGET])
print(f'Correlation of {BASELINE_FEATURE} with {TARGET}: {corr_with_target:.4f}')

In [ ]:
# Now let's build a simple threshold-based classifier using this single feature.
# The idea: pick a cutoff value. If the feature is above (or below) the cutoff,
# predict class 1; otherwise predict class 0.
#
# We'll try a range of thresholds and pick the one that maximises ROC-AUC.
# This is about as simple as a "model" can get — just a single if/else rule.

thresholds = np.linspace(
    train[BASELINE_FEATURE].quantile(0.05),
    train[BASELINE_FEATURE].quantile(0.95),
    200
)

best_auc = 0
best_threshold = None
best_direction = None  # whether higher values predict class 1 or class 0

for t in thresholds:
    # Try both directions: "above threshold = class 1" and "below threshold = class 1"
    preds_above = (train[BASELINE_FEATURE] >= t).astype(int)
    preds_below = (train[BASELINE_FEATURE] < t).astype(int)

    auc_above = roc_auc_score(train[TARGET], preds_above)
    auc_below = roc_auc_score(train[TARGET], preds_below)

    if auc_above > best_auc:
        best_auc = auc_above
        best_threshold = t
        best_direction = 'above'
    if auc_below > best_auc:
        best_auc = auc_below
        best_threshold = t
        best_direction = 'below'

print(f'Best threshold:  {best_threshold:.4f}')
print(f'Best direction:  predict 1 when {BASELINE_FEATURE} is {best_direction} threshold')
print(f'Baseline ROC-AUC (single-feature rule): {best_auc:.4f}')
print(f'\nFor context, a random guess gives ROC-AUC = 0.5000')
print(f'Our simple rule already gets us to {best_auc:.4f} — any real model should beat this.')

---
## 5. Preprocessing

Most ML models need numeric input. We need to:

1. **Encode categorical variables** — convert text categories into numbers
2. **Scale/normalise numeric features** — put features on a similar scale

### Why scale?
Some models (like Logistic Regression, SVMs, neural networks) are sensitive to the magnitude
of features. If one feature ranges from 0-1 and another from 0-1,000,000, the big one can
dominate. **StandardScaler** transforms each feature to have mean=0 and std=1.

Tree-based models (Random Forest, XGBoost, etc.) generally don't need scaling, but it
doesn't hurt — and it's a good habit to learn.

### Important: fit on training data only!
We must fit our scaler/encoder on **training data** and then **transform** both train and test.
Otherwise we'd be leaking information from the test set into our preprocessing.

In [ ]:
# Separate features and target
drop_cols = ['id', TARGET]
X = train.drop(columns=[c for c in drop_cols if c in train.columns])
y = train[TARGET]
X_test_final = test.drop(columns=[c for c in ['id'] if c in test.columns])

print(f'Feature matrix shape: {X.shape}')
print(f'Test matrix shape:    {X_test_final.shape}')

In [ ]:
# Encode categorical columns using LabelEncoder
# LabelEncoder converts each unique string to an integer (e.g. 'RENT'->2, 'OWN'->1, 'MORTGAGE'->0)
#
# Note: For models like Logistic Regression, one-hot encoding is usually better because
# label encoding implies an ordering that may not exist. But for tree-based models,
# label encoding works just fine — and it's simpler to start with.

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Fit on the combined unique values from train + test to avoid unseen labels
    le.fit(pd.concat([X[col], X_test_final[col]]).astype(str))
    X[col] = le.transform(X[col].astype(str))
    X_test_final[col] = le.transform(X_test_final[col].astype(str))
    label_encoders[col] = le
    print(f'  Encoded {col}: {le.classes_}')

print('\nAll categorical columns encoded.')

---
## 6. Train/Test Split

We split our **training data** into two parts:
- **Train split** — used to train the model
- **Validation split** — used to evaluate the model on data it hasn't seen

This gives us a realistic estimate of how the model will perform on truly unseen data
(i.e., the competition's test set).

We use `stratify=y` to ensure both splits have the same ratio of positive/negative examples.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,       # 20% held out for validation
    random_state=42,     # for reproducibility
    stratify=y           # keep class balance consistent in both splits
)

print(f'Training split:   {X_train.shape[0]:,} rows')
print(f'Validation split: {X_val.shape[0]:,} rows')
print(f'\nTarget balance in train: {y_train.mean():.4f}')
print(f'Target balance in val:   {y_val.mean():.4f}')

In [ ]:
# Scale numeric features
# IMPORTANT: fit the scaler on the TRAINING split only, then transform both splits and the test set.

scaler = StandardScaler()

# We only scale the numeric columns (categorical ones are already integers)
cols_to_scale = [c for c in numeric_cols if c in X_train.columns]

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_val[cols_to_scale] = scaler.transform(X_val[cols_to_scale])
X_test_final[cols_to_scale] = scaler.transform(X_test_final[cols_to_scale])

print(f'Scaled {len(cols_to_scale)} numeric columns.')
print(f'\nAfter scaling, training features look like:')
X_train[cols_to_scale].describe().round(2)

---
## 7. Choosing a Performance Metric

Different problems call for different metrics. Common classification metrics include:

| Metric | What it measures | Good when... |
|--------|-----------------|-------------|
| **Accuracy** | Proportion of all predictions that are correct | Classes are balanced |
| **Precision** | Of those predicted positive, how many actually are | False positives are expensive (e.g. spam filter) |
| **Recall** | Of those actually positive, how many we caught | False negatives are expensive (e.g. disease screening) |
| **F1 Score** | Harmonic mean of precision and recall | You need a balance of precision and recall |
| **ROC-AUC** | How well the model ranks positives above negatives, across all thresholds | You care about ranking and the problem is imbalanced |

For this competition, the evaluation metric is **ROC-AUC** (Area Under the ROC Curve).

ROC-AUC measures how well the model **ranks** positive examples above negative ones,
regardless of the threshold you pick. A score of 0.5 = random, 1.0 = perfect.

This is a great metric for imbalanced problems because it doesn't depend on class distribution. For a more detailed explanation: https://www.geeksforgeeks.org/machine-learning/auc-roc-curve/

---
## 8. Train a Model — Logistic Regression

Let's start simple. **Logistic Regression** is a great first model because:
- It's fast
- It's interpretable
- It gives us a sensible probability output
- It sets a solid baseline for more complex models to beat

Logistic Regression models the probability of a binary outcome as a function of
the input features using a logistic (sigmoid) curve. Despite the name, it's a
classification algorithm, not a regression one.

**Further reading:**
- scikit-learn documentation: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
- A visual, intuitive explanation: https://mlu-explain.github.io/logistic-regression/

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# predict_proba gives us probabilities — we want the probability of class 1
lr_val_probs = lr.predict_proba(X_val)[:, 1]
lr_auc = roc_auc_score(y_val, lr_val_probs)

print(f'Logistic Regression — Validation ROC-AUC: {lr_auc:.4f}')

**A quick note...**

Let's just appreciate what just happened. That last cell? Err, that was it. That was a machine learning model! In 4 lines of code we created a model, trained it, made a prediction, then checked the performance. 4 lines.

The coding for most ML models is pretty straightforward. It's the thinking around what we're doing that's important.

Although, that being said, you won't believe how deep this rabbit hole goes...

In [ ]:
# Let's visualise the ROC curve and the confusion matrix side by side

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curve
RocCurveDisplay.from_predictions(y_val, lr_val_probs, ax=axes[0], name='Logistic Regression')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Confusion matrix (using 0.5 threshold)
lr_val_preds = (lr_val_probs >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_val, lr_val_preds, ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix (threshold=0.5)')

plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_val, lr_val_preds))

---
## 9. A Better Model — LightGBM

Logistic Regression is a linear model — it can only learn linear relationships between
features and the target. Real-world data often has **non-linear** patterns.

**Gradient boosting** is a powerful technique that builds many small decision trees
sequentially, where each tree tries to correct the mistakes of the previous ones.
LightGBM is one of the most popular implementations — it's fast, handles large datasets
well, and tends to perform very competitively out of the box.

Unlike scikit-learn's built-in `GradientBoostingClassifier`, LightGBM uses a
histogram-based algorithm that bins continuous features into discrete buckets before
finding splits. This makes it dramatically faster on larger datasets like this one.

**Further reading:**
- LightGBM documentation: https://lightgbm.readthedocs.io/en/latest/
- scikit-learn-compatible API: https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html
- A visual explanation of decision trees: https://mlu-explain.github.io/decision-tree/

**Note:** This model is more computationally expensive than Logistic Regression.
Depending on the size of the dataset and your hardware, it may take a minute or two to train.

In [ ]:
lgb = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)
lgb.fit(X_train, y_train)

lgb_val_probs = lgb.predict_proba(X_val)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_val_probs)

print(f'LightGBM — Validation ROC-AUC: {lgb_auc:.4f}')
print(f'Logistic Regression was:        {lr_auc:.4f}')
print(f'Improvement:                    {lgb_auc - lr_auc:+.4f}')

In [ ]:
# Let's visualise the ROC curve and the confusion matrix side by side

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curve
RocCurveDisplay.from_predictions(y_val, lgb_val_probs, ax=axes[0], name='LightGBM')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Confusion matrix (using 0.5 threshold)
lgb_val_preds = (lgb_val_probs >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_val, lgb_val_preds, ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix (threshold=0.5)')

plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_val, lgb_val_preds))

**How did this go?**

Did it perform better than before? It usually would.

Think (or have a search) about why this model may produce a better result than the logistic regression model. Are there any drawbacks to using this kind of model instead?

In [ ]:
# Feature importance — which features did the model find most useful?
# This is one of the great things about tree-based models: built-in interpretability.

feat_imp = pd.Series(lgb.feature_importances_, index=X_train.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Feature Importance (Gradient Boosting)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### What does this mean?

Have a look at this feature importance plot. What is this telling us?
- could we drop features?
- could we have some problematic co-correlation here?
- does this fit with your domain research? 

---
## 10. Quick Hyperparameter Tuning

Models have **hyperparameters** — settings that control how they learn. These are
different from the parameters the model learns during training (like the coefficients
in Logistic Regression or the tree splits in gradient boosting). Hyperparameters are
set *before* training begins, and finding good values for them can make a big difference.
We call this "Hyperparameter Optimisation" (HPO), or tuning.

For LightGBM, the key hyperparameters we'll tune are:

- `n_estimators` — how many trees to build. More trees = more capacity to learn, but
  also slower and higher risk of overfitting.
- `learning_rate` — how much each new tree contributes. Lower values need more trees
  but often generalise better.
- `max_depth` — how deep each tree can grow. Deeper trees capture more complex patterns
  but can overfit. Set to -1 for no limit.
- `num_leaves` — the maximum number of leaves per tree. This is LightGBM's primary
  control on model complexity (it grows leaf-wise rather than depth-wise). A good
  rule of thumb is to keep `num_leaves` less than `2^max_depth`.

**Further reading on these hyperparameters:**
- LightGBM parameter tuning guide: https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html
- A good practical overview: https://machinelearningmastery.com/configure-gradient-boosting-algorithm/

We'll do a simple manual search over a few combinations. In the real world, you might
use `GridSearchCV`, `RandomizedSearchCV`, or tools like Optuna for a smarter search.

We use **cross-validation** to evaluate each set of hyperparameters. This means splitting
the training data into K folds, training on K-1 and validating on the remaining fold,
then averaging the results. This gives a more robust estimate than a single train/val split.

**Note:** This cell runs multiple models with cross-validation, so it may take several
minutes to complete.

In [ ]:
# Let's try a few combinations and see what works best
# (This might take a minute or two to run)

param_grid = [
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 4, 'num_leaves': 15},
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4, 'num_leaves': 31},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 5, 'num_leaves': 31},
    {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'num_leaves': 63},
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_score = 0
best_params = None

# We use the FULL training features (X, y) for cross-validation now,
# not the train/val split, so we use all available data.
# We need to re-encode and scale the full X again.
X_full = train.drop(columns=[c for c in drop_cols if c in train.columns]).copy()
for col in categorical_cols:
    X_full[col] = label_encoders[col].transform(X_full[col].astype(str))


for params in param_grid:
    model = LGBMClassifier(random_state=42, verbosity=-1, **params)
    scores = cross_val_score(model, X_full, y, cv=cv, scoring='roc_auc')
    mean_score = scores.mean()
    print(f'  {params} => CV ROC-AUC: {mean_score:.4f} (+/- {scores.std():.4f})')

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print(f'\nBest params: {best_params}')
print(f'Best CV ROC-AUC: {best_score:.4f}')

**Why is there no graphs here?**

We've found the most important features earlier. These may vary very slightly with our new best hyperparams, but it's worth noting that the last stage didn't produce "a model". Instead, we just found out which hyperparams are best.

If you like, you can re-run section 9, but inserting the best hyperparams that we find above. This will show you the the best ROC-AUC curve and features importances.

---
## 11. Train Final Model & Generate Submission

Now that we've found our best hyperparameters, we:
1. **Retrain** on ALL the training data (not just the train split)
2. **Predict** probabilities on the test set
3. **Create** a submission file

Why retrain on all the data? Because more data = better model. We've already validated
our hyperparameters using cross-validation, so we're confident in them.

In [ ]:
# Train the final model on all training data with the best hyperparameters
final_model = LGBMClassifier(random_state=42, verbosity=-1, **best_params)
final_model.fit(X_full, y)

# Predict probabilities for the test set
test_probs = final_model.predict_proba(X_test_final)[:, 1]

print(f'Predictions generated for {len(test_probs):,} test rows.')
print(f'Prediction stats: min={test_probs.min():.4f}, mean={test_probs.mean():.4f}, max={test_probs.max():.4f}')

In [ ]:
# Create the submission file
# IMPORTANT: Always check sample_submission.csv to see what format is expected!

submission = sample_sub.copy()
submission[TARGET] = test_probs

submission.to_csv('submission.csv', index=False)
print('Submission saved to submission.csv')
print(f'\nFirst few rows:')
submission.head()

---
## Done! What's Next?

Congratulations — you've built and submitted your first model! Here are some ideas
for improving your score or, more importantly, building some fun models.

### Quick wins

**Try different tree-based models.** We've used LightGBM here, but there are other strong
gradient boosting libraries worth comparing: `xgboost` and `catboost` each have
slightly different approaches to tree building and can sometimes outperform on
different datasets. scikit-learn also provides `RandomForestClassifier` if you
want to try a non-boosting ensemble method. These libraries are pre-installed
on Kaggle, but if you're running locally you'll need to install them:
```
pip install xgboost catboost
```
Then you can use them like:
```python
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
```
A good exercise is to train all three boosting libraries with similar
hyperparameters and compare their validation scores — you'll often find they
produce quite similar results, but small differences can matter at the margins.

**Try a neural network.** If you want to explore deep learning on tabular data,
Keras (part of TensorFlow) lets you build a simple feed-forward neural network
in just a few lines. Neural networks don't always beat gradient boosting on
tabular data, but they're worth experimenting with — and it's a good way to
get familiar with the framework. Plus, it's fun.

It's worth reading up, watching some youtube vids, or asking claude to explain how
neural nets work before jumping in. Like the models built using `scikit-learn`, models 
made with `keras` are pretty easy to code, but also really interesting to learn how they 
work. It also helps to have this understanding to help appreciate what the hyperparams
all mean, and the complexities of explainability with more and more complex networks.

Keras is pre-installed on Kaggle, but locally:
```
pip install tensorflow
```
A minimal example for binary classification:
```python
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
  layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
  layers.Dense(32, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
model.fit(X_train, y_train, epochs=20, batch_size=256, validation_data=(X_val, y_val))
```
This example builds a network that starts with the input dimensions of your training data, `d`
and works towards a single output sigmoid node.

`d -> 64 -> 32 -> 1`

The code defines the network, compliles it, then uses `model.fit` in a simililar way to 
all the other models. The definition and compiling steps are important as the network
architecture can get far far more complex than this with use-cases such as image analysis, 
and therefore a separate script with the model definition can be common.

Note that scaling your features (which we've already done) is particularly
important for neural networks — they tend to struggle with unscaled inputs.
This is also where Kaggle's GPU accelerator will make a real difference to
training speed.

**Tune more hyperparameters**, or use `RandomizedSearchCV` / Optuna for a smarter,
more automated search. Note that different models have different hyperparameters. For each one you try, do some quick research into how they work and what hyperparameters they have.

### Better feature encoding

- **One-hot encoding** — Instead of label encoding (which assigns arbitrary integers),
  one-hot encoding creates a separate binary column for each category. This avoids
  implying a false ordering. Use `pd.get_dummies()` or `OneHotEncoder` from scikit-learn.

- **Ordinal encoding** — Some categorical variables *do* have a natural order. This is
  called **ordinality**. For example, a loan grade of A/B/C/D/E/F/G has a clear ranking
  from best to worst. For variables like this, label encoding is actually reasonable —
  but you should encode them in the correct order (e.g. A=0, B=1, C=2, ...) rather
  than letting `LabelEncoder` assign alphabetical order by accident.
  Use `OrdinalEncoder` from scikit-learn and explicitly pass the category order:
  ```python
  from sklearn.preprocessing import OrdinalEncoder
  oe = OrdinalEncoder(categories=[['A', 'B', 'C', 'D', 'E', 'F', 'G']])
  ```

- **Target encoding** — Replace each category with the mean of the target variable
  for that category. This can be very powerful, but you need to be careful about
  overfitting (use it within cross-validation folds). scikit-learn provides
  `TargetEncoder` for this.

Take a moment to look at the categorical columns in this dataset and think about
which encoding strategy makes sense for each one.

### Feature engineering

- Create new features by combining existing ones (e.g. ratios, interactions)
- Bin continuous variables into categories
- Look at the competition discussion forum for ideas from other competitors

### Advanced techniques

- Ensemble multiple models together (averaging predictions from different model types)
- Use K-fold cross-validation to generate out-of-fold predictions for stacking
- Handle class imbalance with techniques like SMOTE or class weights

Good luck!

## Further notes

**GPU acceleration** — The models in this notebook use scikit-learn, which is
CPU-only. If you switch to `xgboost`, `lightgbm`, or `catboost`, you can take
advantage of Kaggle's free GPU accelerator (Settings > Accelerator > GPU).
However, it won't happen automatically — you need to tell the library to use
the GPU. For example:
```python
# XGBoost
XGBClassifier(device='cuda', ...)

# LightGBM
LGBMClassifier(device='gpu', ...)

# CatBoost
CatBoostClassifier(task_type='GPU', ...)
```
For the dataset sizes in Playground competitions, the difference may be modest,
but it becomes significant when you start training larger models or running
extensive hyperparameter searches.

**Why this won't win the competition** — The model we've built here is solid and
sensible, but it won't trouble the top of the leaderboard. Kaggle competition
winners typically use techniques that squeeze out every last fraction of a
percentage point: deep ensembles of many different model types, extensive
feature engineering tailored to the specific dataset, stacking and blending
strategies, and aggressive hyperparameter tuning over hundreds or thousands of
runs. These approaches are optimised for leaderboard performance, often at the
cost of simplicity, interpretability, and maintainability. In a business context,
the priorities are usually different — you need models that stakeholders can
understand, that are straightforward to deploy and monitor, and that can be
explained to regulators or customers. A well-tuned gradient boosting model with
clear feature importance (like what we've built here) is often exactly what you'd
want in production, even if it wouldn't place first in a competition. Treat
Kaggle as a great learning environment, but keep in mind that the winning
strategies don't always translate to good engineering practice.